本次实验以AAAI 2014会议论文数据为基础，要求实现或调用无监督聚类算法，了解聚类方法。

### 任务介绍
每年国际上召开的大大小小学术会议不计其数，发表了非常多的论文。在计算机领域的一些大型学术会议上，一次就可以发表涉及各个方向的几百篇论文。按论文的主题、内容进行聚类，有助于人们高效地查找和获得所需要的论文。本案例数据来源于AAAI 2014上发表的约400篇文章，由[UCI](https://archive.ics.uci.edu/ml/datasets/AAAI+2014+Accepted+Papers!)公开提供，提供包括标题、作者、关键词、摘要在内的信息，希望大家能根据这些信息，合理地构造特征向量来表示这些论文，并设计实现或调用聚类算法对论文进行聚类。最后也可以对聚类结果进行观察，看每一类都是什么样的论文，是否有一些主题。

基本要求：
1. 将文本转化为向量，实现或调用无监督聚类算法，对论文聚类，例如10类（可使用已有工具包例如sklearn）；
2. 观察每一类中的论文，调整算法使结果较为合理；
3. 无监督聚类没有标签，效果较难评价，因此没有硬性指标，跑通即可，主要让大家了解和感受聚类算法，比较简单。

扩展要求：
1. 对文本向量进行降维，并将聚类结果可视化成散点图。

注：group和topic也不能完全算是标签，因为
1. 有些文章作者投稿时可能会选择某个group/topic但实际和另外group/topic也相关甚至更相关；
2. 一篇文章可能有多个group和topic，作为标签会出现有的文章同属多个类别，这里暂不考虑这样的聚类；
3. group和topic的取值很多，但聚类常常希望指定聚合成出例如5/10/20类；
4. 感兴趣但同学可以思考利用group和topic信息来量化评价无监督聚类结果，不作要求。

提示：
1. 高维向量的降维旨在去除一些高相关性的特征维度，保留最有用的信息，用更低维的向量表示高维数据，常用的方法有PCA和t-SNE等；
2. 降维与聚类是两件不同的事情，聚类实际上在降维前的高维向量和降维后的低维向量上都可以进行，结果也可能截然不同；
3. 高维向量做聚类，降维可视化后若有同一类的点不在一起，是正常的。在高维空间中它们可能是在一起的，降维后损失了一些信息。

## 1. 文本向量化与聚类（基本要求）

把每篇论文的 title、keywords、topics、abstract 拼到一起，用 TF-IDF 转成向量，再用 sklearn 的 KMeans 聚成 10 类。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

df = pd.read_csv('data/[UCI] AAAI-14 Accepted Papers - Papers.csv')
print('论文数:', len(df))

texts = (df['title'].fillna('') + ' '
         + df['keywords'].fillna('').str.replace('\n', ' ') + ' '
         + df['topics'].fillna('').str.replace('\n', ' ') + ' '
         + df['abstract'].fillna(''))

vec = TfidfVectorizer(stop_words='english', max_df=0.5, min_df=2)
X = vec.fit_transform(texts)
print('TF-IDF 特征维度:', X.shape)

km = KMeans(n_clusters=10, random_state=42, n_init=10)
labels = km.fit_predict(X)
print('每类论文数量:', np.bincount(labels))

## 2. 观察每一类中的论文（基本要求）

对每个簇打印 TF-IDF 中心词排名前几的关键词，再随便挑几篇论文的标题看看，大致判断是不是有共同主题。

In [ ]:
terms = vec.get_feature_names_out()
order = km.cluster_centers_.argsort()[:, ::-1]

for i in range(10):
    top_words = [terms[j] for j in order[i, :8]]
    sample_titles = df.loc[labels == i, 'title'].head(3).tolist()
    print(f'--- 第 {i} 类 (共 {(labels == i).sum()} 篇) ---')
    print('top 关键词:', ', '.join(top_words))
    for t in sample_titles:
        print('  -', t)
    print()

## 3. 降维可视化（扩展要求）

TF-IDF 是 2977 维的稀疏向量，没法直接画图。这里先用 PCA 把维度降到 50（去掉冗余），再用 t-SNE 降到 2 维画散点图，用聚类标签上色。

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE

svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X)

tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='pca')
X_2d = tsne.fit_transform(X_svd)

plt.figure(figsize=(8, 6))
for i in range(10):
    mask = labels == i
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], s=15, label=f'cluster {i}')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.title('AAAI-14 papers clustering (t-SNE)')
plt.tight_layout()
plt.show()